In [ ]:
import numpy as np
import pandas as pd
from astropy.timeseries import LombScargle
from scipy.stats import skew, kurtosis, shapiro
from preprocessing import fourier_features, stetson_K, fourier_fit

In [ ]:
def compute_features_to_csv(light_curves, outfile="features.csv"):
    """
    light_curves: list of (t, f) tuples
    outfile     : CSV file to write

    Returns the DataFrame of features.
    """

    rows = []

    for idx, (t, f) in enumerate(light_curves):

        # remove NaNs
        mask = np.isfinite(t) & np.isfinite(f)
        t, f = t[mask], f[mask]

        # --- Period (Lomb–Scargle) ---
        freq, power = LombScargle(t, f).autopower()
        best_period = 1 / freq[np.argmax(power)]

        # --- Flux distribution features ---
        Q1 = np.percentile(f, 25)
        Q3 = np.percentile(f, 75)
        Q31 = Q3 - Q1
        Std = np.std(f)
        gamma1 = skew(f)
        gamma2 = kurtosis(f, fisher=True)
        W, _ = shapiro(f)
        K = stetson_K(f)

        # --- Fourier features ---
        R21, R31, phi21, phi31, Amp = fourier_features(best_period, t, f)

        # store
        rows.append({
            "period": best_period,
            "Q31": Q31,
            "Amp": Amp,
            "W": W,
            "K": K,
            "Std": Std,
            "gamma1": gamma1,
            "gamma2": gamma2,
            "R21": R21,
            "R31": R31,
            "phi21": phi21,
            "phi31": phi31
        })

        print(f"Processed light curve {idx+1}/{len(light_curves)}")

    df = pd.DataFrame(rows)
    df.to_csv(outfile, index=False)
    print(f"\nSaved feature table to: {outfile}")

    return df
